# Create a HF dataset to use with collect_dictionary script

* Create a dataset with prompts for dictionary collection
* Multiple prompt templates and multiple styles (concepts for unlearning)
* Result prompts are a cartesian product of templates and styles
* ~Borrowed from SAeUron paper~
* Using styles that give reasonable results with SD 1.4

In [1]:
from datasets import Dataset, Features, Value
from itertools import product
from huggingface_hub import HfApi

/home/mgarbowski/repos/zzsn-projekt/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Mark the previous version as v1

In [3]:
!hf auth login

Hint: A new version of huggingface_hub (1.20.1) is available! You are using version 1.13.0.
User is already logged in. Use `hf auth login --force` to force re-login.


In [5]:
repo_id = "mgarbowski/zzsn-style-prompts"
api = HfApi()
api.create_tag(repo_id=repo_id, repo_type="dataset", tag="v1", revision="main")

## Create prompts with multiple styles

In [6]:
prompt_templates = [
    "Gothic cathedral with flying buttresses and stained glass windows in {style} style.",
    "A bear dressed as a medieval knight in armor in {style} style.",
    "A bird with feathers as iridescent as an oil slick in the sunlight in {style} style.",
    "A butterfly emerging from a jeweled cocoon in {style} style.",
    "A cat wearing a superhero cape leaping between buildings in {style} style.",
    "A dog wearing aviator goggles piloting an airplane in {style} style.",
    "A goldfish swimming in a crystal-clear bowl in {style} style.",
    "A candle’s flame flickering in a mysterious old library in {style} style.",
    "Flower blooming in the middle of a snow-covered landscape in {style} style.",
    "A frog with a croak that sounds like a jazz musician’s trumpet in {style} style.",
    "Wild horse galloping across the prairie at sunrise in {style} style.",
    "A man hiking through a dense forest in {style} style.",
    "Jellyfish floating serenely in deep blue water in {style} style.",
    "Rabbit peering out from a burrow in {style} style.",
    "A classic BLT sandwich on toasted bread in {style} style.",
    "Sea waves crashing over ancient coastal ruins in {style} style.",
    "Statue of a forgotten hero covered in ivy in {style} style.",
    "Tower soaring above the clouds in {style} style.",
    "A majestic oak tree in a serene forest in {style} style.",
    "Moonlit waterfall in a serene forest in {style} style.",
]

styles = [
    "Van Gogh",
    "Picasso",
    "pencil sketch",
    "mosaic",
    "cyberpunk",
    "cartoon",
]

In [7]:
def make_prompt(template, style):
    return template.format(style=style)


make_prompt(prompt_templates[0], styles[0])

'Gothic cathedral with flying buttresses and stained glass windows in Van Gogh style.'

In [8]:
features = Features(
    {
        "anchor_prompt": Value("string"),
        "style": Value("string"),
        "prompt": Value("string"),
    }
)

items = [
    {
        "anchor_prompt": anchor_prompt,
        "style": style,
        "prompt": make_prompt(anchor_prompt, style),
    }
    for anchor_prompt, style in product(prompt_templates, styles)
]

dataset = Dataset.from_list(items, features=features)

In [9]:
dataset

Dataset({
    features: ['anchor_prompt', 'style', 'prompt'],
    num_rows: 120
})

In [10]:
dataset.push_to_hub("mgarbowski/zzsn-style-prompts")

Setting num_proc from 1 back to 1 for the train split to disable multiprocessing as it only contains one shard.
Creating parquet from Arrow format: 100%|██████████| 1/1 [00:00<00:00, 137.52ba/s]
Processing Files (0 / 0): |          |  0.00B /  0.00B            
Processing Files (1 / 1): 100%|██████████| 5.27kB / 5.27kB, 8.78kB/s  
Processing Files (1 / 1): 100%|██████████| 5.27kB / 5.27kB, 4.39kB/s  
New Data Upload: 100%|██████████| 5.27kB / 5.27kB, 4.39kB/s  
Uploading the dataset shards: 100%|██████████| 1/1 [00:02<00:00,  2.04s/ shards]


CommitInfo(commit_url='https://huggingface.co/datasets/mgarbowski/zzsn-style-prompts/commit/509d7d6b6f44890d8bab2e0470781c166e42fdf8', commit_message='Upload dataset', commit_description='', oid='509d7d6b6f44890d8bab2e0470781c166e42fdf8', pr_url=None, repo_url=RepoUrl('https://huggingface.co/datasets/mgarbowski/zzsn-style-prompts', endpoint='https://huggingface.co', repo_type='dataset', repo_id='mgarbowski/zzsn-style-prompts'), pr_revision=None, pr_num=None)

In [11]:
api.create_tag(repo_id=repo_id, repo_type="dataset", tag="v2", revision="main")